# Nucleotide Transformer v2 finetuning

In [1]:
import os
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

In [2]:
import torch
torch.__version__

'2.1.2+cu121'

## For bacteriophages

In [3]:
DATA_DIR="phage_data/"
DATASET_NAME="phage_clean_dataset"
MODEL_NAME="InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"
OUTPUT_MODEL_NAME="model_weights/nt2-phages-lora"
device="cuda:0"

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
dataset = Dataset.load_from_disk(os.path.join(DATA_DIR, DATASET_NAME))
dataset = Dataset.from_dict({"sequence": dataset[:1000]["sequence"]})
dataset

Dataset({
    features: ['sequence'],
    num_rows: 10000
})

### Get model with LoRA

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME, trust_remote_code=True)

In [6]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    # task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.to(device)

trainable params: 393,216 || all params: 56,298,188 || trainable%: 0.6984523196377119


PeftModel(
  (base_model): LoraModel(
    (model): EsmForMaskedLM(
      (esm): EsmModel(
        (embeddings): EsmEmbeddings(
          (word_embeddings): Embedding(4107, 512, padding_idx=1)
          (dropout): Dropout(p=0.0, inplace=False)
          (position_embeddings): Embedding(2050, 512, padding_idx=1)
        )
        (encoder): EsmEncoder(
          (layer): ModuleList(
            (0-11): 12 x EsmLayer(
              (attention): EsmAttention(
                (self): EsmSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=512, out_features=512, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=512, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_fea

### Train

In [ ]:
# Takes some time... around 23 seconds per 10000 sequences
tokens = tokenizer.batch_encode_plus(dataset[:]["sequence"], return_tensors="pt", padding=True)
tokenized = Dataset.from_dict(tokens)
tokenized

In [ ]:
collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_MODEL_NAME,
    overwrite_output_dir=True,
    per_device_train_batch_size=32,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,     # Higher LR is fine with LoRA
    num_train_epochs=3,
    fp16=True,
    save_steps=5000,
    logging_steps=50,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=collator,
)
trainer.train()

Step,Training Loss


KeyboardInterrupt: 